# Collect Data

In [108]:
%pip install pandas numpy seaborn matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [109]:
import pandas as pd
import numpy as np

In [110]:
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [111]:
df = pd.read_csv('final_data.csv')
df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS
0,4073,10597.4769,4800.2166,56.5923,1355.916,744,33.3,2.02232,40592.3314
1,3702,3245.1189,8032.5534,16.1271,386.394,744,33.3,2.02232,40592.3314
2,3563,0.0000,0.0000,0.0000,0.000,0,33.3,2.02232,40592.3314
3,4128,2116.3271,5439.4737,25.6074,613.536,714,33.3,2.02232,40592.3314
4,3609,1978.3072,5187.5167,16.1911,387.928,712,33.3,2.02232,40592.3314
...,...,...,...,...,...,...,...,...,...
21454,4742,1374.4294,32.1202,0.9751,25.130,741,19.2,2.06677,40529.9208
21455,4722,0.0000,0.0000,0.0000,0.000,0,19.2,2.06677,40529.9208
21456,4609,979.8785,136.3452,1.3825,35.633,736,19.2,2.06677,40529.9208
21457,4639,2460.3767,30.4620,16.0628,413.996,742,19.2,2.06677,40529.9208


In [112]:
from math import log

In [113]:
df['Agua_log'] = df['Agua'].apply(lambda x: log(x) if x > 0 else 0)
df['Gas_log'] = df['Gas'].apply(lambda x: log(x) if x > 0 else 0)
df['Idade2'] = df['Idade'].apply(lambda x: x**2)
df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS,Agua_log,Gas_log,Idade2
0,4073,10597.4769,4800.2166,56.5923,1355.916,744,33.3,2.02232,40592.3314,8.476416,4.035873,16589329
1,3702,3245.1189,8032.5534,16.1271,386.394,744,33.3,2.02232,40592.3314,8.991258,2.780501,13704804
2,3563,0.0000,0.0000,0.0000,0.000,0,33.3,2.02232,40592.3314,0.000000,0.000000,12694969
3,4128,2116.3271,5439.4737,25.6074,613.536,714,33.3,2.02232,40592.3314,8.601438,3.242881,17040384
4,3609,1978.3072,5187.5167,16.1911,387.928,712,33.3,2.02232,40592.3314,8.554010,2.784462,13024881
...,...,...,...,...,...,...,...,...,...,...,...,...
21454,4742,1374.4294,32.1202,0.9751,25.130,741,19.2,2.06677,40529.9208,3.469485,-0.025215,22486564
21455,4722,0.0000,0.0000,0.0000,0.000,0,19.2,2.06677,40529.9208,0.000000,0.000000,22297284
21456,4609,979.8785,136.3452,1.3825,35.633,736,19.2,2.06677,40529.9208,4.915190,0.323893,21242881
21457,4639,2460.3767,30.4620,16.0628,413.996,742,19.2,2.06677,40529.9208,3.416480,2.776506,21520321


In [114]:
target_column = 'Petroleo'
ignore_columns = [
    #'Gas',
    # 'Agua',
    # 'Idade'
]

In [115]:
sample_df = df.sample(frac=1, random_state=42)
sample_df.drop_duplicates(inplace=True)
for col in sample_df.columns:
    Q1 = sample_df[col].quantile(0.25)
    Q3 = sample_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    sample_df = sample_df[(sample_df[col] >= lower_bound) & (sample_df[col] <= upper_bound)]

sample_df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS,Agua_log,Gas_log,Idade2
5527,5636,508.8115,5839.3422,13.8773,414.564,259,22.20,2.03282,39249.2117,8.672373,2.630254,31764496
15902,4453,102.7398,5.0067,3.7128,54.700,676,30.20,2.00144,38798.0539,1.610777,1.311786,19829209
12124,488,7.6646,0.3782,0.2990,4.247,62,30.20,2.00144,38798.0539,-0.972332,-1.207312,238144
1297,6083,125.6144,564.0495,0.2412,7.476,727,30.23,2.12184,36715.5245,6.335142,-1.422129,37002889
2554,3002,69.0662,0.0276,3.0274,22.671,490,28.70,2.04092,36555.2570,-3.589940,1.107704,9012004
...,...,...,...,...,...,...,...,...,...,...,...,...
6265,5463,0.0000,0.0000,0.0000,0.000,0,18.10,2.00677,39989.7428,0.000000,0.000000,29844369
11964,488,102.3119,5.0487,3.9909,56.696,62,30.20,2.00144,38798.0539,1.619131,1.384017,238144
5390,348,578.2746,0.5789,9.2298,264.040,61,24.70,2.01939,40444.9634,-0.546626,2.222437,121104
860,2638,132.0420,3.2312,5.2944,76.693,619,30.20,2.00144,38798.0539,1.172854,1.666650,6959044


In [116]:
X = sample_df.drop(ignore_columns + [target_column], axis=1)
y = sample_df[target_column]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

In [117]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Evaluate Methods

In [118]:
results_df = pd.DataFrame()

def evaluate_and_record(model_name, model, X_train, y_train, X_test, y_test, best_params=None):
    global results_df
    
    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()
    training_time = end_time - start_time
    
    y_pred_treino = model.predict(X_train)
    y_pred_teste = model.predict(X_test)
    
    rmse_treino = np.sqrt(mean_squared_error(y_train, y_pred_treino))
    r2_treino = r2_score(y_train, y_pred_treino)
    mae_treino = mean_absolute_error(y_train, y_pred_treino)
    
    rmse_teste = np.sqrt(mean_squared_error(y_test, y_pred_teste))
    r2_teste = r2_score(y_test, y_pred_teste)
    mae_teste = mean_absolute_error(y_test, y_pred_teste)
    
    new_row = {
        'Modelo': model_name,
        'R2 Treino': r2_treino,
        'R2 Teste': r2_teste,
        'RMSE Treino': rmse_treino,
        'RMSE Teste': rmse_teste,
        'MAE Treino': mae_treino,
        'MAE Teste': mae_teste,
        'Tempo (segundos)': training_time,
        'Hyperparametros': str(best_params) if best_params else 'N/A'
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
    return model

In [119]:
def record_gridsearch_details(grid_search_object, model_prefix):
    global results_df
    
    results = pd.DataFrame(grid_search_object.cv_results_)
    
    for index, row in results.iterrows():
        mean_r2_cv = row['mean_test_score']
        
        mean_fit_time = row['mean_fit_time']
        
        model_name_detail = f"{model_prefix} - Combinacao {index + 1}"
        
        new_row = {
            'Modelo': model_name_detail,
            'R2 Teste': mean_r2_cv,
            'RMSE Teste': np.nan,
            'Tempo (s)': mean_fit_time,
            'Detalhes': f"Combin. Testada (Rank: {row['rank_test_score']})",
            'Hyperparametros': str(row['params'])
        }
        
        results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

# LinearRegression

In [120]:
model = LinearRegression()
evaluate_and_record("Regressao Linear", model, X_train_scaled, y_train, X_test_scaled, y_test)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


# Ridge

In [121]:
alphas = np.logspace(-4, 2, 100)

scoring_metric = 'r2'
ridge_model = Ridge(random_state=42)
grid_search_ridge = GridSearchCV(
    estimator=ridge_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_ridge.fit(X_train_scaled, y_train)

best_alpha_ridge = grid_search_ridge.best_params_['alpha']
best_ridge_model = grid_search_ridge.best_estimator_

In [122]:
evaluate_and_record('Ridge Otimo', best_ridge_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_ridge)

,alpha,np.float64(5.336699231206313)
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,42


# Lasso

In [123]:
lasso_model = Lasso(random_state=42, max_iter=2000) 
grid_search_lasso = GridSearchCV(
    estimator=lasso_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_lasso.fit(X_train_scaled, y_train)

best_alpha_lasso = grid_search_lasso.best_params_['alpha']
best_lasso_model = grid_search_lasso.best_estimator_

In [124]:
evaluate_and_record('Lasso Otimo', best_lasso_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_lasso)

,alpha,np.float64(1.1497569953977356)
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,2000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,42
,selection,'cyclic'


# Random Forest

In [125]:
from sklearn.ensemble import RandomForestRegressor

In [126]:
rf_model = RandomForestRegressor(random_state=42)
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid={
        'n_estimators': [100, 200, 300, 400, 500, 600],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    scoring=scoring_metric,
    cv=3,
    verbose=0,
    n_jobs=-1
)

grid_search_rf.fit(X_train_scaled, y_train)

best_n_estimators_rf = grid_search_rf.best_params_['n_estimators']
best_max_depth_rf = grid_search_rf.best_params_['max_depth']
best_min_samples_split_rf = grid_search_rf.best_params_['min_samples_split']
best_min_samples_leaf_rf = grid_search_rf.best_params_['min_samples_leaf']
best_rf_model = grid_search_rf.best_estimator_

In [127]:
evaluate_and_record('Random Forest Otimo', best_rf_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_rf.best_params_)

,n_estimators,600
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# Gradient Boosting

In [128]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [129]:
from xgboost import XGBRegressor

In [130]:
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)
grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid={
        'n_estimators': [100, 300, 500, 700],
        'learning_rate': [0.01, 0.1, 0.3],
        'max_depth': [3, 5, 7],
        'subsample': [0.7, 1.0],
    },
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search_xgb.fit(X_train_scaled, y_train)

best_n_estimators_xgb = grid_search_xgb.best_params_['n_estimators']
best_max_depth_xgb = grid_search_xgb.best_params_['max_depth']
best_learning_rate_xgb = grid_search_xgb.best_params_['learning_rate']
best_subsample_xgb = grid_search_xgb.best_params_['subsample']
best_xgb_model = grid_search_xgb.best_estimator_

Fitting 3 folds for each of 72 candidates, totalling 216 fits


In [131]:
evaluate_and_record('Gradient Boosting Otimo', best_xgb_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_xgb.best_params_)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


# KNN Regression

In [132]:
from sklearn.neighbors import KNeighborsRegressor

In [133]:
knn_model = KNeighborsRegressor()
grid_search_knn = GridSearchCV(
    estimator=knn_model,
    param_grid={
        'n_neighbors': [3, 5, 7, 9],
        'weights': ['uniform', 'distance'],
        'p': [1, 2]
    },
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_knn.fit(X_train_scaled, y_train)
best_n_neighbors_knn = grid_search_knn.best_params_['n_neighbors']
best_weights_knn = grid_search_knn.best_params_['weights']
best_p_knn = grid_search_knn.best_params_['p']
best_knn_model = grid_search_knn.best_estimator_

In [134]:
evaluate_and_record('KNN Otimo', best_knn_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_knn.best_params_)

,n_neighbors,5
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,1
,metric,'minkowski'
,metric_params,None
,n_jobs,None


# SVR (Suport Vector Regression)

In [135]:
from sklearn.svm import SVR

In [136]:
svr_model = SVR()
grid_search_svr = GridSearchCV(
    estimator=svr_model,
    param_grid={
        'C': [1000, 2000, 3000, 4000, 5000],
        'epsilon': [0.01, 0.1, 1],
        'kernel': ['linear', 'rbf']
    },
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=0,
    n_jobs=-1
)

grid_search_svr.fit(X_train_scaled, y_train)

best_C_svr = grid_search_svr.best_params_['C']
best_epsilon_svr = grid_search_svr.best_params_['epsilon']
best_kernel_svr = grid_search_svr.best_params_['kernel']
best_svr_model = grid_search_svr.best_estimator_

In [137]:
evaluate_and_record('SVR Otimo', best_svr_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_svr.best_params_)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,5000
,epsilon,1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


# Neural Network

In [138]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

In [ ]:
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(random_state=42, max_iter=2000))
])

mlp_param_grid = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 100), (200, 100), 
                                (200, 200), (100, 100, 100)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__solver': ['adam', 'sgd'],
    'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
    'mlp__learning_rate': ['constant', 'adaptive']
}

grid_search_mlp = GridSearchCV(
    estimator=mlp_pipeline,
    param_grid=mlp_param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search_mlp.fit(X_train, y_train)
best_params_mlp = grid_search_mlp.best_params_
best_mlp_model = grid_search_mlp.best_estimator_

Fitting 3 folds for each of 224 candidates, totalling 672 fits


In [ ]:
evaluate_and_record('MLP Otimo', best_mlp_model, X_train, y_train, X_test, y_test, best_params_mlp)

MLP Regressor
Best parameters: {'mlp__activation': 'tanh', 'mlp__alpha': 0.1, 'mlp__hidden_layer_sizes': (200, 200), 'mlp__learning_rate': 'constant', 'mlp__solver': 'adam'}
RMSE: 94.8516 (Treino), 326.4406 (Teste)
MAE: 41.7225 (Treino), 149.3877 (Teste)
R2: 0.9946 (Treino), 0.9268 (Teste)


# GridSearch Details

In [ ]:
record_gridsearch_details(grid_search_ridge, "Ridge GridSearch")
record_gridsearch_details(grid_search_lasso, "Lasso GridSearch")
record_gridsearch_details(grid_search_rf, "Random Forest GridSearch")
record_gridsearch_details(grid_search_xgb, "XGBoost GridSearch")
record_gridsearch_details(grid_search_knn, "KNN GridSearch")
record_gridsearch_details(grid_search_svr, "SVR GridSearch")
record_gridsearch_details(grid_search_mlp, "MLP GridSearch")

In [ ]:
results_df

In [ ]:
results_df.to_csv('model_comparison_results.csv', index=False)